In [1]:
import os
from datasets import load_dataset

dataset_name = "SQuAD_2.0"

val_path = os.path.join(os.getcwd(), "Datasets", dataset_name, "validation-00000-of-00001.parquet")

dataset = load_dataset("parquet", data_files={'val': val_path})
dataset

DatasetDict({
    val: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 11873
    })
})

In [2]:
reduced_test_set = dataset["val"].shuffle(seed=42).select(range(5500, 11000))
data_batch_size = 500
data_batches = [reduced_test_set.select(range(i,i+data_batch_size)) for i in range(0, len(reduced_test_set), data_batch_size)]

In [3]:
import os
from transformers import AutoTokenizer

output_name = "t1_bert"
output_dir = os.path.join(os.getcwd(), "models", output_name, "checkpoint-24705")

tokenizer = AutoTokenizer.from_pretrained(output_dir)

In [4]:
max_length = 384
stride = 128

def preprocess_validation_examples(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = inputs.pop("overflow_to_sample_mapping")
    example_ids = []

    for i in range(len(inputs["input_ids"])):
        sample_idx = sample_map[i]
        example_ids.append(examples["id"][sample_idx])

        sequence_ids = inputs.sequence_ids(i)
        offset = inputs["offset_mapping"][i]
        inputs["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None for k, o in enumerate(offset)
        ]

    inputs["example_id"] = example_ids
    return inputs

In [5]:
tokenized_inputs = []
for batch in data_batches:
    tokenized_test = batch.map(
        preprocess_validation_examples,
        batched=True,
        remove_columns=batch.column_names,
    )

    tokenized_inputs.append(tokenized_test)

tokenized_inputs[0]

Dataset({
    features: ['input_ids', 'attention_mask', 'offset_mapping', 'example_id'],
    num_rows: 509
})

In [6]:
import collections

example_mappings=[]

for tokenized_test in tokenized_inputs:
    example_to_features = collections.defaultdict(list)
    for idx, feature in enumerate(tokenized_test):
        example_to_features[feature["example_id"]].append(idx)
    
    example_mappings.append(example_to_features)

In [7]:
import torch
from transformers import AutoModelForQuestionAnswering

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trained_model = AutoModelForQuestionAnswering.from_pretrained(output_dir).to(device)

In [8]:
from tqdm import tqdm
from torch.utils.data import DataLoader
import torch.nn.functional as F

batch_size = 32
n_best = 5

max_answer_length = 30
no_answer_threshold = 0.05

def post_process(tokenized_test, data_batch, example_to_features):
    t_tokenized_test = tokenized_test.remove_columns(["example_id", "offset_mapping"])
    t_tokenized_test.set_format("torch")

    test_dataloader = DataLoader(t_tokenized_test, batch_size=batch_size, shuffle=False)    # no need for shuffling

    start_probs = torch.empty((len(t_tokenized_test), 384), device=device)
    end_probs = torch.empty((len(t_tokenized_test), 384), device=device)

    trained_model.eval()  # Set the model to evaluation mode
    print("model inputing")
    with torch.no_grad():
        for i, batch in enumerate(test_dataloader):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = trained_model(**batch)

            batch_start_probs = F.softmax(outputs.start_logits, dim=-1)
            batch_end_probs = F.softmax(outputs.end_logits, dim=-1)

            start_idx = i * batch_size
            end_idx = start_idx + batch_start_probs.shape[0]

            start_probs[start_idx:end_idx] = batch_start_probs
            end_probs[start_idx:end_idx] = batch_end_probs
    
    no_answer_probs = start_probs[:, 0] * end_probs[:, 0]

    start_indexes = torch.argsort(start_probs, dim=-1, descending=True)[:, :n_best] # get n best scores for each example
    end_indexes = torch.argsort(end_probs, dim=-1, descending=True)[:, :n_best]

    best_start_probs = torch.gather(start_probs, dim=1, index=start_indexes)
    best_end_probs = torch.gather(end_probs, dim=1, index=end_indexes)

    best_probs = best_start_probs.unsqueeze(2) * best_end_probs.unsqueeze(1)    # calcalate the n_best squared best probabilities for each feature

    max_prob_indices = torch.argsort(best_probs.view((len(t_tokenized_test), -1)), dim=1, descending=True)

    print("extracting answers")
    predicted_answers = []
    for example in tqdm(data_batch):
        example_id = example["id"]
        context = example["context"]
        answers = []    # track all answers for this id

        # go through each feature that is associated to that example
        for feature_index in example_to_features[example_id]:
            offsets = tokenized_test["offset_mapping"][feature_index]   # get offset mapping
            probability_indicies = max_prob_indices[feature_index]      # get n_best squared indexes

            no_answer_probability = no_answer_probs[feature_index].cpu().item()

            # if the no answer probability exceeds threshold, give an empty answer
            if no_answer_probability > no_answer_threshold:
                answers.append(
                    {
                        "text": "",
                        "prob_score": no_answer_probability,
                        "no_answer_probability": no_answer_probability,
                    }
                )
                break
            
            # iterate through the indicies, going from maximum probability first
            for flat_idx in probability_indicies:
                start_index = start_indexes[feature_index, flat_idx // n_best]
                end_index = end_indexes[feature_index, flat_idx % n_best]

                # if the highest probability index is not part of context then move to next highest probability
                if offsets[start_index] is None or offsets[end_index] is None:
                    continue
                if (end_index < start_index or end_index - start_index + 1> max_answer_length):
                    continue
                # only append one set of probabilities per feature, where the prob score is the highest
                answers.append(
                    {
                        "text": context[offsets[start_index][0] : offsets[end_index][1]],
                        "prob_score": best_probs[feature_index, flat_idx // n_best, flat_idx % n_best].cpu().item(),
                        "no_answer_probability": no_answer_probability,
                    }
                )
                break

        if len(answers) == 0:
            predicted_answers.append({"id": example_id, "prediction_text": "", "no_answer_probability": 1.0})
        else:
            # check for the best answer for each id
            best_answer = max(answers, key=lambda x: x["prob_score"])
            predicted_answers.append({"id": example_id, "prediction_text": best_answer["text"], "no_answer_probability": best_answer["no_answer_probability"]})
        
    return predicted_answers

In [9]:
predicted_answers = []

for tokenized_test, data_batch, example_mapping in zip(tokenized_inputs, data_batches, example_mappings):
    pp = post_process(tokenized_test, data_batch, example_mapping)
    predicted_answers = predicted_answers + pp

model inputing
extracting answers


100%|██████████| 500/500 [03:57<00:00,  2.10it/s]


model inputing
extracting answers


100%|██████████| 500/500 [04:07<00:00,  2.02it/s]


model inputing
extracting answers


...

100%|██████████| 500/500 [03:50<00:00,  2.17it/s]


model inputing
extracting answers


100%|██████████| 500/500 [03:49<00:00,  2.18it/s]


model inputing
extracting answers


100%|██████████| 500/500 [03:46<00:00,  2.21it/s]


In [11]:
references = [{"id": ex["id"], "answers": ex["answers"]} for ex in reduced_test_set]

In [12]:
import evaluate
metric = evaluate.load("squad_v2")
metric.compute(predictions=predicted_answers, references=references)

{'exact': 66.10909090909091,
 'f1': 69.34055372959114,
 'total': 5500,
 'HasAns_exact': 61.54127100073046,
 'HasAns_f1': 68.03252210107787,
 'HasAns_total': 2738,
 'NoAns_exact': 70.63721940622737,
 'NoAns_f1': 70.63721940622737,
 'NoAns_total': 2762,
 'best_exact': 66.43636363636364,
 'best_exact_thresh': 0.018038932234048843,
 'best_f1': 69.51196192923446,
 'best_f1_thresh': 0.045763809233903885}